# Refutation tests and sensitivity

Run from the repository root, with the raw OULAD files in `raw/` (see README).

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
for p in [root, *root.parents]:
    if (p / "raw").is_dir():
        root = p
        break
BASE = str(root)

In [1]:
!pip install -q econml dowhy

import os, json, time, math, warnings
import numpy as np, pandas as pd, joblib, networkx as nx
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.exceptions import DataConversionWarning
from econml.dml import CausalForestDML
from dowhy import CausalModel

warnings.filterwarnings('ignore', category=DataConversionWarning)

CKPT = f'{BASE}/refutation_checkpoints'
os.makedirs(CKPT, exist_ok=True)

SEED = 42
N_SIMULATIONS = 100
FORCE_RERUN = False

CANON_ATE = 0.004786482048864249

cf_canonical = joblib.load(f'{BASE}/cf_canonical_model.pkl')
full = pd.read_csv(f'{BASE}/oulad_full_corpus.csv')

confounders_v2 = ['region', 'imd_band', 'highest_education', 'age_band', 'gender',
                  'disability', 'code_presentation', 'num_of_prev_attempts',
                  'studied_credits', 'code_module']
categorical_v2 = [c for c in confounders_v2
                  if c not in ('num_of_prev_attempts', 'studied_credits')]

n_missing = int(full['imd_band'].isna().sum())
full['imd_band'] = full['imd_band'].fillna('Missing')
print(f'imd_band NaN filled to "Missing": {n_missing}   (expect 830)')

X_encoded_v2 = pd.get_dummies(full[confounders_v2],
                              columns=categorical_v2, drop_first=True).astype(float)
assert X_encoded_v2.shape == (17529, 40), f'shape drift: {X_encoded_v2.shape}'

ate_rebuilt = float(cf_canonical.ate(X_encoded_v2))
print(f'ATE on rebuilt matrix : {ate_rebuilt!r}')
print(f'canonical             : {CANON_ATE!r}')
assert abs(ate_rebuilt - CANON_ATE) < 1e-6, 'REBUILD MISMATCH — stop, do not proceed'
print('rebuild verified to 1e-6\n')

T = full['oucontent_clicks'].values
Y = full['performance_gain'].values

confounder_cols = list(X_encoded_v2.columns)
df_refute = X_encoded_v2.copy()
df_refute['oucontent_clicks'] = T
df_refute['performance_gain'] = Y

G = nx.DiGraph()
for col in confounder_cols:
    G.add_edge(col, 'oucontent_clicks')
    G.add_edge(col, 'performance_gain')
G.add_edge('oucontent_clicks', 'performance_gain')

model = CausalModel(data=df_refute, treatment='oucontent_clicks',
                    outcome='performance_gain', graph=G,
                    effect_modifiers=confounder_cols)
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.econml.dml.CausalForestDML",
    effect_modifiers=confounder_cols,
    method_params={"init_params": {
        'model_y': HistGradientBoostingRegressor(random_state=SEED),
        'model_t': HistGradientBoostingRegressor(random_state=SEED),
        'discrete_treatment': False, 'honest': True,
        'min_samples_leaf': 50, 'n_estimators': 500, 'cv': 5, 'random_state': SEED
    }, "fit_params": {}})

ate_dowhy = float(np.mean(np.asarray(estimate.value).ravel()))
print(f'DoWhy-wrapped estimate: {ate_dowhy!r}')
assert abs(ate_dowhy - CANON_ATE) < 1e-6, 'DoWhy refit diverged from canonical'

SD_T = float(full['oucontent_clicks'].std())
SD_Y = float(full['performance_gain'].std())
VAR_T = SD_T ** 2
print(f'\nSD(T) = {SD_T:.4f}   SD(Y) = {SD_Y:.4f}')

def _extract(ref):
    out = {'estimated_effect': float(np.mean(np.asarray(ref.estimated_effect).ravel()))}
    try:
        out['new_effect'] = float(np.mean(np.asarray(ref.new_effect).ravel()))
    except Exception:
        out['new_effect'] = float('nan')
    rr = getattr(ref, 'refutation_result', None)
    if isinstance(rr, dict) and rr.get('p_value') is not None:
        out['p_value'] = float(np.ravel(rr['p_value'])[0])
        out['significant'] = rr.get('is_statistically_significant')
    else:
        out['p_value'] = float('nan')
        out['significant'] = None
    return out

def run_refuter(key, legacy_name, **kwargs):
    path = f'{CKPT}/A_{key}.json'
    if os.path.exists(path) and not FORCE_RERUN:
        rec = json.load(open(path))
        print(f'[cached] {key} — {rec["minutes"]:.1f} min on the original run\n')
        print(rec['summary'])
        return rec
    print(f'[START] {key}  num_simulations={kwargs.get("num_simulations")}')
    print('        expect roughly 37 minutes. Do not close the tab.', flush=True)
    t0 = time.time()
    np.random.seed(SEED)
    ref = model.refute_estimate(identified_estimand, estimate, **kwargs)
    mins = (time.time() - t0) / 60
    rec = {'refuter': key, 'num_simulations': kwargs.get('num_simulations'),
           'minutes': round(mins, 2), 'summary': str(ref)}
    rec.update(_extract(ref))
    json.dump(rec, open(path, 'w'), indent=2)
    with open(f'{BASE}/{legacy_name}', 'w') as f:
        f.write(str(ref))
    print(f'\n{ref}')
    print(f'\n[DONE] {mins:.1f} min  ->  {path}')
    return rec

print('\nsetup complete — Part A is ready to run.')

DoWhy-wrapped estimate: 0.004786482048864249

SD(T) = 781.7294   SD(Y) = 19.5537

setup complete — Part A is ready to run.


In [ ]:
import os, json, time, tempfile
import numpy as np

def _atomic_write(text, path):
    d = os.path.dirname(path) or '.'
    fd, tmp = tempfile.mkstemp(dir=d, suffix='.tmp')
    try:
        with os.fdopen(fd, 'w') as f:
            f.write(text)
            f.flush()
            try:
                os.fsync(f.fileno())
            except OSError:
                pass
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise

def _load_ckpt(path):
    """Record, or None if missing / empty / unparseable."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return None
    try:
        with open(path) as f:
            return json.load(f)
    except (json.JSONDecodeError, ValueError):
        return None

def _extract(ref):
    out = {'estimated_effect': float(np.mean(np.asarray(ref.estimated_effect).ravel()))}
    try:
        out['new_effect'] = float(np.mean(np.asarray(ref.new_effect).ravel()))
    except Exception:
        out['new_effect'] = float('nan')

    rr = getattr(ref, 'refutation_result', None)
    if isinstance(rr, dict) and rr.get('p_value') is not None:
        out['p_value'] = float(np.ravel(rr['p_value'])[0])
        sig = rr.get('is_statistically_significant')

        out['significant'] = None if sig is None else bool(np.ravel(sig)[0])
    else:
        out['p_value'] = float('nan')
        out['significant'] = None
    return out

def run_refuter(key, legacy_name, **kwargs):
    path = f'{CKPT}/A_{key}.json'

    rec = None if FORCE_RERUN else _load_ckpt(path)
    if rec is not None:
        print(f'[cached] {key} — {rec.get("minutes")} min originally\n')
        print(rec['summary'])
        return rec

    if os.path.exists(path):
        os.remove(path)
        print(f'[warn] discarded unreadable checkpoint: {path}')

    print(f'[START] {key}  num_simulations={kwargs.get("num_simulations")}')
    print('        expect roughly 37 minutes. Do not close the tab.', flush=True)

    t0 = time.time()
    np.random.seed(SEED)
    ref = model.refute_estimate(identified_estimand, estimate, **kwargs)
    mins = (time.time() - t0) / 60

    _atomic_write(str(ref), f'{BASE}/{legacy_name}')
    print(f'\n{ref}')

    rec = {'refuter': key, 'num_simulations': kwargs.get('num_simulations'),
           'minutes': round(mins, 2), 'summary': str(ref)}
    rec.update(_extract(ref))
    _atomic_write(json.dumps(rec, indent=2, default=str), path)

    print(f'\n[DONE] {mins:.1f} min  ->  {path}')
    return rec

_probe = {'significant': np.bool_(True), 'p': np.float64(0.5), 'n': np.int64(100)}
json.dumps(_probe, default=str)
assert json.loads(json.dumps({'significant': bool(np.ravel(np.bool_(True))[0])}))['significant'] is True
print('serialisation self-test passed — numpy types will no longer abort the write.\n')

r_placebo = run_refuter('placebo', 'refute_1_placebo_n100.txt',
                        method_name='placebo_treatment_refuter',
                        placebo_type='permute', num_simulations=N_SIMULATIONS)

serialisation self-test passed — numpy types will no longer abort the write.

[START] placebo  num_simulations=100
        expect roughly 37 minutes. Do not close the tab.

Refute: Use a Placebo Treatment
Estimated effect:0.004786482048864249
New effect:3.749717824886508e-06
p value:0.96


[DONE] 41.2 min  ->  ./refutation_checkpoints/A_placebo.json


In [ ]:
r_random = run_refuter('random_common_cause', 'refute_2_random_cause_n100.txt',
                       method_name='random_common_cause',
                       num_simulations=N_SIMULATIONS)

[START] random_common_cause  num_simulations=100
        expect roughly 37 minutes. Do not close the tab.

Refute: Add a random common cause
Estimated effect:0.004786482048864249
New effect:0.00480145594210773
p value:0.6399999999999999


[DONE] 43.0 min  ->  ./refutation_checkpoints/A_random_common_cause.json


In [ ]:
r_subset = run_refuter('data_subset', 'refute_3_subset_n100.txt',
                       method_name='data_subset_refuter',
                       subset_fraction=0.8, num_simulations=N_SIMULATIONS)

[START] data_subset  num_simulations=100
        expect roughly 37 minutes. Do not close the tab.

Refute: Use a subset of data
Estimated effect:0.004786482048864249
New effect:0.004808984841818245
p value:0.8400000000000001


[DONE] 32.3 min  ->  ./refutation_checkpoints/A_data_subset.json


In [2]:
import os, gc, json, time
import numpy as np, pandas as pd
from econml.dml import CausalForestDML
from sklearn.ensemble import HistGradientBoostingRegressor

N_BOOT = 100
SAVE_EVERY = 5
PROG = f'{CKPT}/A_bootstrap_progress.json'

T_obs = full['oucontent_clicks'].to_numpy(dtype=float)
Y_obs = full['performance_gain'].to_numpy(dtype=float)
X_mat = X_encoded_v2.to_numpy(dtype=float)
n = len(T_obs)

done = []
if os.path.exists(PROG) and os.path.getsize(PROG) > 0:
    try:
        done = json.load(open(PROG))['estimates']
        print(f'resuming — {len(done)} of {N_BOOT} iterations already complete\n')
    except Exception:
        done = []

resume_from = len(done)
t_start = time.time()
for i in range(resume_from, N_BOOT):
    rng = np.random.default_rng(SEED + i)
    idx = rng.integers(0, n, size=n)

    cf_b = CausalForestDML(
        model_y=HistGradientBoostingRegressor(random_state=SEED),
        model_t=HistGradientBoostingRegressor(random_state=SEED),
        discrete_treatment=False, n_estimators=500, min_samples_leaf=50,
        honest=True, cv=5, random_state=SEED)
    cf_b.fit(Y_obs[idx], T_obs[idx], X=X_mat[idx], W=None)
    done.append(float(cf_b.ate(X_mat)))

    del cf_b
    gc.collect()

    n_this_session = i + 1 - resume_from
    el = time.time() - t_start
    per = el / max(n_this_session, 1)
    eta = per * (N_BOOT - i - 1)
    print(f'  [{i+1:>3}/{N_BOOT}]  ATE={done[-1]:+.6f}   '
          f'{per:5.1f}s/fit   elapsed {el/60:5.1f} min   ETA {eta/60:5.1f} min',
          flush=True)

    if (i + 1) % SAVE_EVERY == 0 or i + 1 == N_BOOT:
        json.dump({'estimates': done}, open(PROG, 'w'), indent=2)

est = np.array(done)
mean, sd = float(est.mean()), float(est.std(ddof=1))

med = float(np.median(est))
tail = ((est > CANON_ATE).mean() if CANON_ATE > med else (est < CANON_ATE).mean())
p_emp = float(min(1.0, 2 * tail))

summary = (f'Refute: Bootstrap Sample Dataset (instrumented replacement)\n'
           f'Estimated effect:{CANON_ATE}\n'
           f'New effect:{mean}\n'
           f'p value:{p_emp}')

rec = {'refuter': 'bootstrap', 'num_simulations': N_BOOT,
       'minutes': round((time.time() - t_start) / 60, 2), 'resumed_from': resume_from, 'summary': summary,
       'estimated_effect': CANON_ATE, 'new_effect': mean,
       'p_value': p_emp, 'significant': bool(p_emp < 0.05),
       'bootstrap_sd': sd,
       'bootstrap_ci': [float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))],
       'INSTRUMENTED_REPLACEMENT': True}

_atomic_write(summary, f'{BASE}/refute_4_bootstrap_n100.txt')
_atomic_write(json.dumps(rec, indent=2, default=str), f'{CKPT}/A_bootstrap.json')

print('\n' + summary)
print(f'\nbootstrap SD        = {sd:.6f}')
print(f'bootstrap 2.5–97.5% = [{rec["bootstrap_ci"][0]:.6f}, {rec["bootstrap_ci"][1]:.6f}]')
print(f'attenuation vs canonical = {100*(1-mean/CANON_ATE):.1f}%   (v1 saw 43.3%)')
print(f'\n[DONE] {rec["minutes"]:.1f} min')

resuming — 95 of 100 iterations already complete

  [ 96/100]  ATE=+0.004850    23.4s/fit   elapsed   0.4 min   ETA   1.6 min
  [ 97/100]  ATE=+0.004596    20.3s/fit   elapsed   0.7 min   ETA   1.0 min
  [ 98/100]  ATE=+0.004849    19.0s/fit   elapsed   1.0 min   ETA   0.6 min
  [ 99/100]  ATE=+0.004332    18.2s/fit   elapsed   1.2 min   ETA   0.3 min
  [100/100]  ATE=+0.004683    18.0s/fit   elapsed   1.5 min   ETA   0.0 min


In [3]:
import os, json, tempfile
import numpy as np

def _atomic_write(text, path):
    d = os.path.dirname(path) or '.'
    fd, tmp = tempfile.mkstemp(dir=d, suffix='.tmp')
    try:
        with os.fdopen(fd, 'w') as f:
            f.write(text)
            f.flush()
            try:
                os.fsync(f.fileno())
            except OSError:
                pass
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise

est = np.array(json.load(open(f'{CKPT}/A_bootstrap_progress.json'))['estimates'])
assert len(est) == 100, f'expected 100 estimates, found {len(est)}'

mean, sd = float(est.mean()), float(est.std(ddof=1))
half_p = float(np.mean((est > CANON_ATE) + 0.5 * (est == CANON_ATE)))
p_emp = float(2 * min(half_p, 1 - half_p))
lo, hi = float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

summary = (f'Refute: Bootstrap Sample Dataset (pure resampling, instrumented)\n'
           f'Estimated effect:{CANON_ATE}\n'
           f'New effect:{mean}\n'
           f'p value:{p_emp}')

rec = {'refuter': 'bootstrap', 'num_simulations': 100, 'minutes': None,
       'summary': summary,
       'estimated_effect': CANON_ATE, 'new_effect': mean,
       'p_value': p_emp, 'significant': bool(p_emp < 0.05),
       'bootstrap_sd': sd, 'bootstrap_ci': [lo, hi],
       'INSTRUMENTED_REPLACEMENT': True,
       'note': ('Pure row-resampling bootstrap. NOT DoWhy BootstrapRefuter, which '
                'additionally injects measurement error into every confounder '
                '(noise=0.1 x column SD by default).')}

_atomic_write(summary, f'{BASE}/refute_4_bootstrap_n100.txt')
_atomic_write(json.dumps(rec, indent=2, default=str), f'{CKPT}/A_bootstrap.json')

print(summary)
print(f'\n--- distribution of 100 bootstrap ATEs ---')
print(f'  mean            {mean!r}')
print(f'  canonical       {CANON_ATE!r}')
print(f'  deviation       {100*(mean/CANON_ATE - 1):+.2f}%')
print(f'  SD              {sd:.6f}')
print(f'  2.5–97.5%       [{lo:.6f}, {hi:.6f}]')
print(f'  min / max       {est.min():.6f} / {est.max():.6f}')
print(f'  below 0.0030    {int((est < 0.0030).sum())} of 100')
print(f'  analytic CI     [0.0026, 0.0070]   (canonical, for comparison)')

print(f'\n--- versus DoWhy BootstrapRefuter ---')
print(f'  DoWhy (v1, n=5/8)   ~0.002715   p=0.0    -43.3%')
print(f'  pure resampling     {mean:.6f}   p={p_emp}   {100*(mean/CANON_ATE-1):+.1f}%')
print('\n  These test different things. DoWhy\'s refuter resamples AND adds')
print('  N(0, 0.1 x SD) measurement error to every confounder — its docstring')
print('  says so explicitly. The attenuation is a confounder-corruption result,')
print('  not a resampling-variance result.')

Refute: Bootstrap Sample Dataset (pure resampling, instrumented)
Estimated effect:0.004786482048864249
New effect:0.00470529405573643
p value:0.88

--- distribution of 100 bootstrap ATEs ---
  mean            0.00470529405573643
  canonical       0.004786482048864249
  deviation       -1.70%
  SD              0.000373
  2.5–97.5%       [0.003903, 0.005415]
  min / max       0.003609 / 0.005530
  below 0.0030    0 of 100
  analytic CI     [0.0026, 0.0070]   (canonical, for comparison)

--- versus DoWhy BootstrapRefuter ---
  DoWhy (v1, n=5/8)   ~0.002715   p=0.0    -43.3%
  pure resampling     0.004705   p=0.88   -1.7%

  These test different things. DoWhy's refuter resamples AND adds
  N(0, 0.1 x SD) measurement error to every confounder — its docstring
  says so explicitly. The attenuation is a confounder-corruption result,
  not a resampling-variance result.


In [ ]:
r_bootstrap = run_refuter('bootstrap', 'refute_4_bootstrap_n100.txt',
                          method_name='bootstrap_refuter',
                          num_simulations=N_SIMULATIONS)

[START] bootstrap  num_simulations=100
        expect roughly 37 minutes. Do not close the tab.


In [4]:
rows = []
for k in ['placebo', 'random_common_cause', 'data_subset', 'bootstrap']:
    p = f'{CKPT}/A_{k}.json'
    if os.path.exists(p):
        rows.append(json.load(open(p)))
    else:
        print(f'MISSING: {k} — that refuter has not run')

if rows:
    refutation_table = pd.DataFrame(rows)[
        ['refuter', 'num_simulations', 'estimated_effect', 'new_effect',
         'p_value', 'significant', 'minutes']]
    refutation_table['pct_of_original'] = 100 * refutation_table['new_effect'] / CANON_ATE
    refutation_table.to_csv(f'{BASE}/refutation_n100.csv', index=False)
    print(refutation_table.to_string(index=False))
    print(f'\ntotal refutation runtime: {refutation_table["minutes"].sum():.0f} min')

            refuter  num_simulations  estimated_effect  new_effect  p_value  significant  minutes  pct_of_original
            placebo              100          0.004786    0.000004     0.96        False    41.18         0.078340
random_common_cause              100          0.004786    0.004801     0.64        False    43.00       100.312837
        data_subset              100          0.004786    0.004809     0.84        False    32.31       100.470132
          bootstrap              100          0.004786    0.004705     0.88        False      NaN        98.303807

total refutation runtime: 116 min


In [5]:
path = f'{CKPT}/B_literal_grid.json'
if os.path.exists(path) and not FORCE_RERUN:
    print('[cached]'); print(json.load(open(path))['summary'])
else:
    np.random.seed(SEED)
    ref_lit = model.refute_estimate(
        identified_estimand, estimate,
        method_name='add_unobserved_common_cause',
        confounders_effect_on_treatment='linear',
        confounders_effect_on_outcome='linear',
        effect_strength_on_treatment=[0.01, 0.02, 0.05, 0.1],
        effect_strength_on_outcome=[0.01, 0.02, 0.05, 0.1])
    json.dump({'summary': str(ref_lit)}, open(path, 'w'), indent=2)
    print(ref_lit)

print(f"\nmax analytic bias at corner (0.1, 0.1) = {(0.1*0.1)/VAR_T:.3e}")
print(f"  as a share of ATE                    = {100*((0.1*0.1)/VAR_T)/CANON_ATE:.6f}%")
print(f"  kappa_t*kappa_y required to null ATE = {CANON_ATE*VAR_T:,.1f}")
print("  -> the grid sits ~4 orders of magnitude below the treatment scale.")

[cached]
Refute: Add an Unobserved Common Cause
Estimated effect:0.004786482048864249
New effect:(np.float64(0.004769644209066673), np.float64(0.004837166694331631))


max analytic bias at corner (0.1, 0.1) = 1.636e-08
  as a share of ATE                    = 0.000342%
  kappa_t*kappa_y required to null ATE = 2,925.0
  -> the grid sits ~4 orders of magnitude below the treatment scale.


In [6]:
K_GRID = [0.0, 0.05, 0.075, 0.09, 0.10, 0.11, 0.125, 0.15,
          0.175, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75]

def _fresh_cf():
    return CausalForestDML(
        model_y=HistGradientBoostingRegressor(random_state=SEED),
        model_t=HistGradientBoostingRegressor(random_state=SEED),
        discrete_treatment=False, n_estimators=500, min_samples_leaf=50,
        honest=True, cv=5, random_state=SEED)

T_obs = full['oucontent_clicks'].to_numpy(dtype=float)
Y_obs = full['performance_gain'].to_numpy(dtype=float)
X_mat = X_encoded_v2.to_numpy(dtype=float)

rows_c = []
for k in K_GRID:
    hit = [f for f in os.listdir(CKPT)
           if f.startswith('C_k_') and abs(float(f[4:-5]) - k) < 1e-9]
    if hit and not FORCE_RERUN:
        rows_c.append(json.load(open(f'{CKPT}/{hit[0]}')))
        print(f'[cached] k={k:.3f}')
        continue
    rng = np.random.default_rng(SEED)
    U = rng.standard_normal(len(T_obs))
    t0 = time.time()
    cf_k = _fresh_cf()
    cf_k.fit(Y_obs - (k*SD_Y)*U, T_obs + (k*SD_T)*U, X=X_mat, W=None)
    ate_k = float(cf_k.ate(X_mat))
    try:
        lo, hi = cf_k.ate_interval(X_mat, alpha=0.05)
        lo, hi = float(np.ravel(lo)[0]), float(np.ravel(hi)[0])
    except Exception:
        lo = hi = float('nan')
    rec = {'k': k, 'kappa_t': k*SD_T, 'kappa_y': k*SD_Y, 'ate_k': ate_k,
           'ci_low': lo, 'ci_high': hi, 'pct_of_original': 100*ate_k/CANON_ATE,
           'minutes': round((time.time()-t0)/60, 2)}
    json.dump(rec, open(f'{CKPT}/C_k_{k:.3f}.json', 'w'), indent=2)
    rows_c.append(rec)
    print(f'[run] k={k:.3f}  ATE_k={ate_k:+.6f}  ci_low={lo:+.6f}', flush=True)

sensitivity_curve = pd.DataFrame(rows_c).sort_values('k').reset_index(drop=True)
sensitivity_curve.to_csv(f'{BASE}/sensitivity_sd_units.csv', index=False)
print()
print(sensitivity_curve[['k','ate_k','ci_low','ci_high','pct_of_original']].to_string(index=False))

[cached] k=0.000
[cached] k=0.050
[cached] k=0.075
[cached] k=0.090
[cached] k=0.100
[cached] k=0.110
[cached] k=0.125
[cached] k=0.150
[cached] k=0.175
[cached] k=0.200
[cached] k=0.300
[cached] k=0.400
[cached] k=0.500
[cached] k=0.600
[cached] k=0.750

    k     ate_k    ci_low   ci_high  pct_of_original
0.000  0.004786  0.002576  0.006997       100.000000
0.050  0.004342  0.001934  0.006749        90.709876
0.075  0.003634  0.000932  0.006336        75.920577
0.090  0.003015  0.000060  0.005970        62.986441
0.100  0.002375 -0.001044  0.005795        49.624430
0.110  0.001871 -0.001831  0.005573        39.084275
0.125  0.001060 -0.002921  0.005041        22.148681
0.150 -0.000540 -0.005225  0.004145       -11.282487
0.175 -0.002078 -0.007248  0.003093       -43.404033
0.200 -0.003450 -0.009455  0.002555       -72.073187
0.300 -0.007256 -0.012845 -0.001667      -151.594787
0.400 -0.009487 -0.012786 -0.006188      -198.202104
0.500 -0.011118 -0.013911 -0.008326      -232.284997
0.

In [7]:
from scipy.stats import kruskal, mannwhitneyu

full['cate_v2'] = cf_canonical.effect(X_encoded_v2)

imd_order = ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%',
             '60-70%', '70-80%', '80-90%', '90-100%', 'Missing']
present_levels = [lvl for lvl in imd_order if lvl in full['imd_band'].unique()]
print("imd_band levels actually present:", full['imd_band'].unique())

full['imd_band'] = pd.Categorical(full['imd_band'], categories=present_levels, ordered=True)
print("\nMean CATE by imd_band:")
print(full.groupby('imd_band', observed=True)['cate_v2'].agg(['mean', 'std', 'count']).reindex(present_levels))

groups_imd = [full.loc[full['imd_band']==lvl, 'cate_v2'].values for lvl in present_levels]
stat_imd, p_imd = kruskal(*[g for g in groups_imd if len(g) > 0])
k_imd = len([g for g in groups_imd if len(g) > 0])
eps_imd = (stat_imd - k_imd + 1) / (len(full) - k_imd)
print(f"\nKruskal-Wallis: H={stat_imd:.2f}, p={p_imd:.4g}, epsilon-squared={eps_imd:.5f}")

edu_order = ['No Formal quals', 'Lower Than A Level', 'A Level or Equivalent',
             'HE Qualification', 'Post Graduate Qualification']
full['highest_education'] = pd.Categorical(full['highest_education'], categories=edu_order, ordered=True)
print("\nMean CATE by highest_education (corrected model):")
print(full.groupby('highest_education', observed=True)['cate_v2'].agg(['mean', 'std', 'count']).reindex(edu_order))

groups_edu = [full.loc[full['highest_education']==lvl, 'cate_v2'].values for lvl in edu_order]
stat_edu, p_edu = kruskal(*groups_edu)
eps_edu = (stat_edu - 5 + 1) / (len(full) - 5)
print(f"\nKruskal-Wallis: H={stat_edu:.2f}, p={p_edu:.4g}, epsilon-squared={eps_edu:.5f}")

full[['id_student', 'code_module', 'code_presentation', 'cate_v2', 'imd_band', 'highest_education']].to_csv(
    BASE + '/heterogeneity_checks_v2.csv', index=False)
print("\nSaved heterogeneity_checks_v2.csv")

imd_band levels actually present: ['80-90%' '90-100%' '50-60%' '40-50%' '60-70%' '20-30%' '70-80%' '10-20'
 '30-40%' 'Missing' '0-10%']

Mean CATE by imd_band:
              mean       std  count
imd_band                           
0-10%     0.005067  0.002004   1438
10-20     0.005348  0.002121   1630
20-30%    0.004690  0.001973   1755
30-40%    0.004986  0.002050   1869
40-50%    0.004685  0.002038   1694
50-60%    0.004741  0.002076   1737
60-70%    0.004529  0.002049   1637
70-80%    0.004974  0.002188   1703
80-90%    0.004705  0.002275   1642
90-100%   0.004524  0.002240   1594
Missing   0.004041  0.002248    830

Kruskal-Wallis: H=442.06, p=1.033e-88, epsilon-squared=0.02466

Mean CATE by highest_education (corrected model):
                                 mean       std  count
highest_education                                     
No Formal quals              0.004758  0.001994    121
Lower Than A Level           0.005313  0.001975   5897
A Level or Equivalent        0.004792

In [8]:
missing_count = full['imd_band'].isna().sum()
print(f"Students silently excluded from the imd_band check: {missing_count}")

imd_order_v2 = ['0-10%', '10-20', '20-30%', '30-40%', '40-50%', '50-60%',
                '60-70%', '70-80%', '80-90%', '90-100%', 'Missing']

full['imd_band'] = pd.Categorical(full['imd_band'], categories=imd_order_v2, ordered=True)

full['imd_band'] = full['imd_band'].fillna('Missing')

print("\nMean CATE by imd_band (corrected, Missing category included):")
print(full.groupby('imd_band', observed=True)['cate_v2'].agg(['mean', 'std', 'count']).reindex(imd_order_v2))

groups_imd_v2 = [full.loc[full['imd_band']==lvl, 'cate_v2'].values for lvl in imd_order_v2]
stat_v2, p_v2 = kruskal(*groups_imd_v2)
eps_v2 = (stat_v2 - 11 + 1) / (len(full) - 11)
print(f"\nKruskal-Wallis (corrected): H={stat_v2:.2f}, p={p_v2:.4g}, epsilon-squared={eps_v2:.5f}")

Students silently excluded from the imd_band check: 0

Mean CATE by imd_band (corrected, Missing category included):
              mean       std  count
imd_band                           
0-10%     0.005067  0.002004   1438
10-20     0.005348  0.002121   1630
20-30%    0.004690  0.001973   1755
30-40%    0.004986  0.002050   1869
40-50%    0.004685  0.002038   1694
50-60%    0.004741  0.002076   1737
60-70%    0.004529  0.002049   1637
70-80%    0.004974  0.002188   1703
80-90%    0.004705  0.002275   1642
90-100%   0.004524  0.002240   1594
Missing   0.004041  0.002248    830

Kruskal-Wallis (corrected): H=442.06, p=1.033e-88, epsilon-squared=0.02466
